## Transform Data - Cleaning the cities_df and save into new df called cities_df_cleaned

In [8]:
from api_project.ETL.transform_data import TransformData

transform_data = TransformData()
df = transform_data.load_reports_df("reports_USA_2020-04-16")
df.head()

2026-04-30 12:33:12,205 | INFO | Loading raw API JSON from C:\Users\rafae\PycharmProjects\Data_Pipelines\api_project\raw_data\covid_api.json


,date,confirmed,deaths,recovered,confirmed_diff,deaths_diff,recovered_diff,last_update,active,active_diff,fatality_rate,iso,region_name,province,region_lat,region_long,region_cities
0,2020-04-16,11057,579,0,115,27,0,2020-04-16 23:30:51,10478,88,0.0524,USA,US,Washington,47.4009,-121.4905,"[{'name': 'Adams', 'date': '2020-04-16', 'fips..."
1,2020-04-16,25734,1072,0,1141,123,0,2020-04-16 23:30:51,24662,1018,0.0417,USA,US,Illinois,40.3495,-88.9861,"[{'name': 'Adams', 'date': '2020-04-16', 'fips..."
2,2020-04-16,27677,956,0,991,96,0,2020-04-16 23:30:51,26721,895,0.0345,USA,US,California,36.1162,-119.6816,"[{'name': 'Alameda', 'date': '2020-04-16', 'fi..."
3,2020-04-16,4237,150,0,273,8,0,2020-04-16 23:30:51,4087,265,0.0354,USA,US,Arizona,33.7298,-111.4312,"[{'name': 'Apache', 'date': '2020-04-16', 'fip..."
4,2020-04-16,223691,14832,0,9237,3215,0,2020-04-16 23:30:51,208859,6022,0.0663,USA,US,New York,42.1657,-74.9481,"[{'name': 'Albany', 'date': '2020-04-16', 'fip..."


In [9]:
df.columns

Index(['date', 'confirmed', 'deaths', 'recovered', 'confirmed_diff',
       'deaths_diff', 'recovered_diff', 'last_update', 'active', 'active_diff',
       'fatality_rate', 'iso', 'region_name', 'province', 'region_lat',
       'region_long', 'region_cities'],
      dtype='str')

In [16]:
df["region_cities"].iloc[0]

[{'name': 'Adams',
  'date': '2020-04-16',
  'fips': 53001,
  'lat': '46.98299757',
  'long': '-118.56017340000001',
  'confirmed': 40,
  'deaths': 0,
  'confirmed_diff': 1,
  'deaths_diff': 0,
  'last_update': '2020-04-16 23:30:51'},
 {'name': 'Asotin',
  'date': '2020-04-16',
  'fips': 53003,
  'lat': '46.18894415',
  'long': '-117.2022851',
  'confirmed': 10,
  'deaths': 0,
  'confirmed_diff': 4,
  'deaths_diff': 0,
  'last_update': '2020-04-16 23:30:51'},
 {'name': 'Benton',
  'date': '2020-04-16',
  'fips': 53005,
  'lat': '46.23946995',
  'long': '-119.51208340000001',
  'confirmed': 283,
  'deaths': 34,
  'confirmed_diff': 17,
  'deaths_diff': 5,
  'last_update': '2020-04-16 23:30:51'},
 {'name': 'Chelan',
  'date': '2020-04-16',
  'fips': 53007,
  'lat': '47.87046092',
  'long': '-120.6173956',
  'confirmed': 63,
  'deaths': 5,
  'confirmed_diff': 2,
  'deaths_diff': 0,
  'last_update': '2020-04-16 23:30:51'},
 {'name': 'Clallam',
  'date': '2020-04-16',
  'fips': 53009,
  'lat

In [34]:
import pandas as pd

base_cols = ["iso", "region_name", "province", "date"]  # keep what you need
cities_col = "region_cities"  # dict of persons data

cities_df = df[base_cols + [cities_col]].explode(cities_col, ignore_index=True)

# drop rows where the list was empty
cities_df = cities_df.dropna(subset=[cities_col])

# expand each city dict into columns
city_details = pd.json_normalize(cities_df[cities_col])
if "date" in city_details.columns:
    city_details = city_details.rename(columns={"date": "city_date"})

cities_df = pd.concat([cities_df.drop(columns=[cities_col]), city_details], axis=1)

# remove duplicate columns
cities_df_cleaned = cities_df.drop(columns=["region_name", "city_date", "last_update" ])

#rename column
cities_df_cleaned = cities_df_cleaned.rename(columns={"iso": "country"})
cities_df_cleaned

,country,province,date,name,fips,lat,long,confirmed,deaths,confirmed_diff,deaths_diff
0,USA,Washington,2020-04-16,Adams,53001.0,46.98299757,-118.56017340000001,40,0,1,0
1,USA,Washington,2020-04-16,Asotin,53003.0,46.18894415,-117.2022851,10,0,4,0
2,USA,Washington,2020-04-16,Benton,53005.0,46.23946995,-119.51208340000001,283,34,17,5
3,USA,Washington,2020-04-16,Chelan,53007.0,47.87046092,-120.6173956,63,5,2,0
4,USA,Washington,2020-04-16,Clallam,53009.0,48.04754642,-123.92263190000001,14,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
2629,USA,South Dakota,2020-04-16,Turner,46125.0,43.3109081,-97.14865776,5,0,0,0
2630,USA,South Dakota,2020-04-16,Unassigned,90046.0,NaN,NaN,0,0,0,0
2631,USA,South Dakota,2020-04-16,Union,46127.0,42.83112163,-96.6557828,4,0,0,0
2632,USA,South Dakota,2020-04-16,Walworth,46129.0,45.43019636,-100.03075140000001,5,0,0,0


In [36]:
# save the cities_df_cleaned to a csv file
transform_data.save_df(cities_df_cleaned, "cities_df_cleaned.csv")


2026-04-30 13:07:59,075 | INFO | Saving DataFrame to C:\Users\rafae\PycharmProjects\Data_Pipelines\api_project\processed_data\cities_df_cleaned.csv


WindowsPath('C:/Users/rafae/PycharmProjects/Data_Pipelines/api_project/processed_data/cities_df_cleaned.csv')

In [37]:
cities_df_cleaned.columns

Index(['country', 'province', 'date', 'name', 'fips', 'lat', 'long',
       'confirmed', 'deaths', 'confirmed_diff', 'deaths_diff'],
      dtype='str')